# 🍥 Sreeshanth Portfolio Builder & Server

This notebook will build the assets (copy images and decode the profile image) and then guide you to run the local server.

In [ ]:
import os
import shutil
import json
import base64
import re

src_dir = r"C:\Users\namir\.gemini\antigravity-ide\brain\afc459ba-159b-4ed9-9ab8-aad1077de185"
dest_dir = r"c:\Users\namir\Downloads\Resume\assets"

os.makedirs(dest_dir, exist_ok=True)

naruto_shop_src = os.path.join(src_dir, "media__1783575978389.jpg")
naruto_nature_src = os.path.join(src_dir, "naruto_nature_hero_1783576887366.png")

shutil.copy(naruto_shop_src, os.path.join(dest_dir, "naruto_shop.jpg"))
shutil.copy(naruto_nature_src, os.path.join(dest_dir, "naruto_nature.png"))
print("Copied images successfully.")

logs_path = os.path.join(src_dir, ".system_generated", "logs", "transcript_full.jsonl")
dest_path = os.path.join(dest_dir, "profile.jpg")

found = False
with open(logs_path, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, 1):
        if "/9j/4AAQSkZ" in line:
            print(f"Found base64 signature in log line {line_num}!")
            try:
                data = json.loads(line)
                content = data.get("content", "")
                
                match = re.search(r"<USER_REQUEST>\s*(.*?)\s*</USER_REQUEST>", content, re.DOTALL)
                if match:
                    b64_data = match.group(1).strip()
                else:
                    b64_data = content.replace("<USER_REQUEST>", "").replace("</USER_REQUEST>", "").strip()
                
                b64_data = "".join(b64_data.split())
                
                with open(dest_path, "wb") as out_f:
                    out_f.write(base64.b64decode(b64_data))
                print("Decoded and wrote profile.jpg successfully.")
                found = True
                break
            except Exception as e:
                print(f"Error decoding: {e}")

if not found:
    print("Could not find the base64 profile image in the logs.")

## Start Web Server

Run the following cell to start a local python server. You can access the portfolio at [http://localhost:8000](http://localhost:8000).

In [ ]:
import http.server
import socketserver

PORT = 8000
Handler = http.server.SimpleHTTPRequestHandler

with socketserver.TCPServer(("", PORT), Handler) as httpd:
    print(f"Serving at port {PORT}. Open http://localhost:8000 in your browser.")
    try:
        httpd.serve_forever()
    except KeyboardInterrupt:
        print("Server stopped.")